# Data Audit

Prepare the repo inside Google Colab and inspect the locally generated Gemini-backed prompt inventory and split manifests. Data generation and augmentation are expected to happen before the repo is pushed and cloned here.

**Runtime:** CPU (no GPU needed)

In [1]:
import os
import subprocess
import sys
from getpass import getpass
from pathlib import Path

try:
    from google.colab import userdata as colab_userdata
except ImportError:
    colab_userdata = None


def read_secret(name: str) -> str:
    if colab_userdata is not None:
        try:
            return colab_userdata.get(name).strip()
        except Exception:
            pass
    return os.environ.get(name, "").strip()


REPO_URL = "https://github.com/aaliyan1230/false-refusal-suppression.git"
HF_TOKEN = read_secret("HF_TOKEN") or getpass("Enter your HF token (or press Enter to skip): ")
REPO_DIR = Path("/content/false-refusal-suppression")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[train,dev]"], check=True)

os.chdir(REPO_DIR)
src_path = REPO_DIR / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(REPO_DIR)
print("HF token loaded:", bool(HF_TOKEN))

/content/false-refusal-suppression
HF token loaded: True


In [2]:
PILOT_PROMPTS = Path("data/processed/prompts/pilot_prompts_gemini.jsonl")
split_dir = Path("data/processed/splits/pilot_gemini")

required_paths = [
    PILOT_PROMPTS,
    split_dir / "discovery.jsonl",
    split_dir / "selection.jsonl",
    split_dir / "holdout.jsonl",
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing prebuilt local data artifacts. Generate them locally before running Kaggle: " + ", ".join(missing)
    )

print(PILOT_PROMPTS)
print(split_dir)

data/processed/prompts/pilot_prompts_gemini.jsonl
data/processed/splits/pilot_gemini


In [3]:
import json
import pandas as pd

records = []
with open(PILOT_PROMPTS, "r", encoding="utf-8") as handle:
    for line in handle:
        line = line.strip()
        if line:
            records.append(json.loads(line))

df = pd.DataFrame(records)
display(df.groupby(["group", "topic"]).size().reset_index(name="count"))
display(df.groupby("family_id").size().reset_index(name="examples_per_family"))
display(
    pd.DataFrame(
        {
            "split": ["discovery", "selection", "holdout"],
            "path": [
                str(split_dir / "discovery.jsonl"),
                str(split_dir / "selection.jsonl"),
                str(split_dir / "holdout.jsonl"),
            ],
        }
    )
)

,group,topic,count
0,benign_borderline,cloud_security_operations,8
1,benign_borderline,defensive_security_testing,8
2,benign_borderline,llm_security,8
3,benign_borderline,shell_process_management,8
4,benign_easy,git_basics,8
5,benign_easy,networking_basics,8
6,benign_easy,observability,8
7,benign_easy,python_basics,8
8,capability_holdout,algorithms,8
9,capability_holdout,containers,8


,family_id,examples_per_family
0,borderline_key_rotation,8
1,borderline_kill_process,8
2,borderline_prompt_injection,8
3,borderline_sql_injection_demo,8
4,easy_dns,8
5,easy_git_status,8
6,easy_logging,8
7,easy_python_dict,8
8,holdout_binary_search,8
9,holdout_docker,8


,split,path
0,discovery,data/processed/splits/pilot_gemini/discovery.j...
1,selection,data/processed/splits/pilot_gemini/selection.j...
2,holdout,data/processed/splits/pilot_gemini/holdout.jsonl
